In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, matthews_corrcoef, roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from collections import defaultdict


import warnings
warnings.filterwarnings('ignore')

MULTI_HOT_ENCODE = True
ONE_HOT_ENCODE = False

In [2]:
def fold_indices(df, n_folds=5):
    f = defaultdict(list)
    unique_combinations = df['drug_combo'].unique()
    np.random.shuffle(unique_combinations)

    for idx, combo in enumerate(unique_combinations):
        row_idx = df.loc[df['drug_combo'] == combo].index
        fold = idx % n_folds
        f[fold].extend(row_idx)

    return f

In [3]:
drug_df = pd.read_csv('data/DrugCombDB/drug_combination_processed.csv')
drug_df = drug_df.dropna(subset=['synergy'])
print(drug_df.head())

  Drug1         Drug2  cell  drug1_db  drug2_db  synergy  synergistic
0  5-FU    BORTEZOMIB    29        35       423  -2.3950            0
1  5-FU     DASATINIB    29        35        98   1.5075            1
2  5-FU     ERLOTINIB    29        35       491   8.2525            1
3  5-FU  GELDANAMYCIN    29        35       507   6.0575            1
4  5-FU     LAPATINIB    29        35       411   4.9200            1


In [4]:
if MULTI_HOT_ENCODE:
    drug_df['drug1_db'] = drug_df['drug1_db'].astype(str)
    drug_df['drug2_db'] = drug_df['drug2_db'].astype(str)
    drug_df['cell'] = drug_df['cell'].astype(str)

    drug_df['drug_combo'] = drug_df.apply(lambda row: tuple(sorted([row['drug1_db'], row['drug2_db']])), axis=1)
    mlb = MultiLabelBinarizer()
    drug_encoded = pd.DataFrame(mlb.fit_transform(drug_df['drug_combo']), columns=mlb.classes_)

    encoder_cell = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    cell_features = encoder_cell.fit_transform(drug_df[['cell']])
    cell_feature_names = encoder_cell.get_feature_names_out(['cell'])
    cell_encoded_df = pd.DataFrame(cell_features, columns=cell_feature_names)

    X = pd.concat([drug_encoded, cell_encoded_df], axis=1)
    y = drug_df['synergistic']

    #X.columns = X.columns.astype(str)
elif ONE_HOT_ENCODE:
    drug_df['drug_combo'] = drug_df.apply(lambda row: '+'.join(map(str, sorted([row['drug1_db'], row['drug2_db']]))), axis=1)

    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_features = encoder.fit_transform(drug_df[['drug_combo', 'cell']])
    feature_names = encoder.get_feature_names_out(['drug_combo', 'cell'])

    encoded_df = pd.DataFrame(encoded_features, columns=feature_names)

    encoded_df.columns = encoded_df.columns.astype(str)

    X = encoded_df
    y = drug_df['synergistic']
else:
    X = drug_df[['drug1_db', 'drug2_db', 'cell']]
    y = drug_df['synergistic']

In [5]:
print(X.shape)
print(y.shape)

scoring = ['accuracy', 'precision', 'recall', 'roc_auc', 'average_precision', 'f1', 'matthews_corrcoef']


(66088, 645)
(66088,)


In [6]:
num_folds=5
folds = fold_indices(drug_df, n_folds=num_folds)

custom_cv_splits = []
for fold_id in range(num_folds):
    test_fold = fold_id
    train_folds = [f for f in range(num_folds) if f != test_fold]

    test_idx = folds[test_fold]
    train_idx = [idx for f in train_folds for idx in folds[f]]

    custom_cv_splits.append((train_idx, test_idx))

rf_classifier_cv = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=30,
    min_samples_leaf=10,
    max_features='sqrt',
    bootstrap=True,
    max_samples=0.7,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

results = cross_validate(
    rf_classifier_cv,
    X,
    y,
    cv=custom_cv_splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [8]:
results_df = pd.DataFrame(results)
# results_df.to_csv('/Users/keremaras/Projects/Temp/plot_cv/log/DrugCombDB/random_forest/rf_cv_drugcombodb.csv')
results_df

,fit_time,score_time,test_accuracy,train_accuracy,test_precision,train_precision,test_recall,train_recall,test_roc_auc,train_roc_auc,test_average_precision,train_average_precision,test_f1,train_f1,test_matthews_corrcoef,train_matthews_corrcoef
0,7.913773,0.396334,0.645702,0.665431,0.580304,0.606282,0.732644,0.743552,0.711053,0.739802,0.648674,0.692901,0.647636,0.667937,0.309203,0.344945
1,7.939312,0.370112,0.644541,0.661822,0.589000,0.599396,0.722315,0.748930,0.716251,0.737459,0.666557,0.689823,0.648881,0.665871,0.302777,0.340438
2,7.923362,0.371413,0.642793,0.668313,0.591603,0.606746,0.705462,0.743974,0.690877,0.741486,0.634799,0.692247,0.643535,0.668389,0.295609,0.350573
3,7.918805,0.411089,0.657596,0.658708,0.592316,0.597533,0.742386,0.751432,0.726199,0.733574,0.671630,0.687667,0.658914,0.665704,0.332153,0.335242
4,7.911947,0.375030,0.639654,0.665602,0.575630,0.603895,0.777205,0.748593,0.715478,0.739350,0.667820,0.694100,0.661400,0.668504,0.309452,0.346842


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [9]:
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [10]:
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=500, n_jobs=-1,
                       random_state=42)

In [11]:
y_pred = rf_classifier.predict(X_test)
mcc = matthews_corrcoef(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
roc_auc_sc = roc_auc_score(y_test, y_pred)
print("ROC AUC Score:", roc_auc_sc)
print("Matthews Correlation Coefficient:", mcc)
print("Classification Report:\n", classification_rep)

ROC AUC Score: 0.7707902298850575
Matthews Correlation Coefficient: 0.5481019809417834
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.70      0.75       400
           1       0.75      0.84      0.79       435

    accuracy                           0.77       835
   macro avg       0.78      0.77      0.77       835
weighted avg       0.78      0.77      0.77       835

